In the following set of experiments, we aim to generate meaningful benchmarks for the dose escalation methods we would like to study.

### Utilities

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime
from doseescalation.dose_escalator import (
    CRMDoseEscalator, 
    DoseEscalatorBase,
    ThreePlusThreeDoseEscalator, 
    UCBDoseEscalator,
    SEEDADoseEscalator,
    SEEDAOriginalDoseEscalator,
    SEEDAPlateauDoseEscalator,
    SEEDAPlateauFixedDoseEscalator,
    SEEDAPlateauNaiveDoseEscalator
)
from doseescalation.estimator import (
    AveragingEstimator
)
from doseescalation.evaluate import (
    plot_dose_proposals, 
    plot_acc_progression,
    plot_n_dles,
    simulate
)
from doseescalation.simulated_env import SimulatedEnv
from typing import Callable, Sequence

/opt/anaconda3/envs/ucl/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [ ]:
# Get the current timestamp for saving results:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [ ]:
def a_key(a):
    return f"a = {a:.1f}"

def cohort_key(cohort):
    return f"Cohort {cohort + 1}"

In [ ]:
def dose_toxic_curve(dose_levels, a_hat):
    return np.power((np.tanh(dose_levels) + 1) / 2, a_hat)

def inv_dose_toxic(p_dle, a):
    return np.arctanh(2 * np.power(p_dle, 1 / a) - 1)

In [ ]:
# Define hyperparameter a:
A_STAR = 1.0

# Dose-toxicity curve (Table 2 from SEEDA paper):
TOXICITY_PROBS = [0.01, 0.05, 0.15, 0.2, 0.45, 0.6]

# Dose-efficacy curve is the per-patient response ~ Bernoulli(q_dose),
# rising then plateauing (shared across all toxicity scenarios):
EFFICACY_PROBS = [0.1, 0.35, 0.6, 0.6, 0.6, 0.6]

# Simulation settings:
N_LEVELS = 6
P_DLE_LEVELS = {
    A_STAR: TOXICITY_PROBS
}
DOSE_LEVELS = {
    a: [inv_dose_toxic(v, a) for v in vs] 
    for a, vs in P_DLE_LEVELS.items()
}
N_A = len(DOSE_LEVELS.keys())
TTL = 0.35
N_TRIALS = 1000
COHORT_SIZE = 3

# Number of patients who showed positive response must be <= COHORT_SIZE (≈ 33% efficacy rate):
N_EFFICATE = 1

# Efficacy env: reuse SimulatedEnv with "dose levels" = indices so the curve maps
# index -> probability, giving n_efficate ~ Binomial(cohort, EFFICACY_PROBS[idx]).
EFFICACY_ENV = SimulatedEnv(list(range(N_LEVELS)), lambda i: EFFICACY_PROBS[int(i)])

# To add more algorithms, extend the following list to include them.
# Both SEEDA variants (current + UCB's original) and all three SEEDA-Plateau
# variants (UCB's original / modified-L1 fix / paper's naive rule) are benchmarked here.
ALGOS = [
    "3 + 3", "CRM", "UCB",
    "SEEDA", "SEEDA (UCB)",
    "SEEDA Plateau (UCB)", "SEEDA Plateau (Modified L1)", "SEEDA Plateau (Paper)",
]

# UCB parameter:
UCB_COEFF = 0.1

# SEEDA and SEEDA Plateau parameters:
P_HAT = (0.02, 0.06, 0.12, 0.20, 0.30, 0.40)
Q_HAT = (0.12, 0.20, 0.30, 0.40, 0.50, 0.59)
ETA = 2
SEEDA_UCB_COEFF = 2.1

# k* = lowest safe dose (p <= theta) with maximal efficacy. Safe doses are 1-4
# (idx 0-3); max efficacy 0.6 is reached at doses 3 & 4 -> k* = dose 3 (index 2).
# Toxicity MTD (highest safe dose) = dose 4 (index 3).
OPTIMAL_DOSE = 2
TOX_MTD = 3
OPTIMAL_DOSES = {a_key(A_STAR): OPTIMAL_DOSE}
CORRECT_MTDS  = {a_key(A_STAR): TOX_MTD}

# Every algorithm is scored against the toxicity MTD.
"""CORRECT_DOSES = {
    a_key(a): {
        algo: CORRECT_MTDS[a_key(a)]
        for algo in ALGOS
    } for a in DOSE_LEVELS
}"""

# Every design is scored & highlighted against the optimal biological dose k* (dose 3).
CORRECT_DOSES = {
    a_key(a): {algo: OPTIMAL_DOSE for algo in ALGOS}
    for a in DOSE_LEVELS
}

# Display-only (1-indexed) version of CORRECT_DOSES, matching the 1-6 dose
# numbering used in plots/tables. Internal dose indices stay 0-indexed
# everywhere else, since that's what the escalators/simulate() emit.
CORRECT_DOSES_DISPLAY = {
    scenario: {algo: dose + 1 for algo, dose in algos.items()}
    for scenario, algos in CORRECT_DOSES.items()
}

In [ ]:
def run_simulations(
    dose_escalator: DoseEscalatorBase,
    dose_levels: Sequence[float],
    dose_toxic_curve: Callable,
    cohort_size: int, 
    n_cohorts: int,
    n_efficate: int = 0,
    efficacy_env=None,
):
    env = SimulatedEnv(dose_levels, dose_toxic_curve)
    return simulate(
        cohort_sizes=[cohort_size] * n_cohorts, 
        dose_escalator=dose_escalator, 
        env=env,
        n_efficate=n_efficate,
        efficacy_env=efficacy_env
    )

In [ ]:
def build_results_table(rec_map, alloc_map, correct_mtds, n_levels, algos,
                        opt_doses=None, efficacy_algos=()):
    records = []
    for scenario in rec_map:
        for algo in algos:
            # Efficacy-aware methods are scored against the efficacy-optimal dose:
            if opt_doses is not None and algo in efficacy_algos:
                correct = opt_doses[scenario]
            # Toxicity-only methods are scored against the toxicity MTD:
            else:
                correct = correct_mtds[scenario]
            recs = np.asarray(rec_map[scenario][algo])
            allocs = np.asarray(alloc_map[scenario][algo])
            for dose in range(n_levels):
                records.append({
                    "Scenario": scenario,
                    "Algorithm": algo,
                    "Dose": dose,
                    # Whether this dose is the algorithm's correct (target) dose:
                    "Is correct": dose == correct,
                    "Rec (in %)": round(100 * np.mean(recs == dose), 2) if recs.size else np.nan,
                    "Alloc (in %)": round(100 * np.mean(allocs == dose), 2) if allocs.size else np.nan
                })
    return pd.DataFrame.from_records(records)

In [ ]:
def _correct_and_majority_styles(pivot_df, correct_doses_dict, bg_color):
    """
    Build a styles DataFrame (same shape as `pivot_df`) where, per row:
      - the correct dose (toxicity MTD) cell gets a `bg_color` background, and
      - the majority dose (the row's argmax: most recommended / allocated) is bold.
    A cell can receive both if the correct dose was also the majority dose.
    """
    styles = pd.DataFrame('', index=pivot_df.index, columns=pivot_df.columns)
    for (scenario, algo) in pivot_df.index:
        row = pivot_df.loc[(scenario, algo)]
        # Background on the correct dose (the MTD we score against):
        correct = correct_doses_dict[scenario][algo]
        if correct in pivot_df.columns:
            styles.loc[(scenario, algo), correct] += f'background-color: {bg_color}; '
        # Bold on the dose this algorithm chose most often:
        if row.notna().any():
            majority = row.idxmax()
            styles.loc[(scenario, algo), majority] += 'font-weight: bold; '
    return styles


def highlight_correct_dose(pivot_df, correct_doses_dict):
    """
    Styler for an on-screen pivot table (index (Scenario, Algorithm), columns =
    dose levels):
      - green background -> the correct dose (toxicity MTD), and
      - bold text        -> the dose most often recommended / allocated.
    """
    styles = _correct_and_majority_styles(
        pivot_df, correct_doses_dict, '#2ca02c82'  # translucent green
    )
    return pivot_df.style.apply(lambda _: styles, axis=None)


def export_table_latex(pivot_df, correct_doses_dict, path, caption=None, label=None):
    """
    Export a dose-level pivot to a LaTeX table at `path`, preserving the
    highlighting: the correct dose (MTD) cell is shaded green and the majority
    (most-chosen) dose cell is bold. Requires these packages in the document:
        \\usepackage[table]{xcolor}   % \\cellcolor
        \\usepackage{booktabs}        % \\toprule / \\midrule / \\bottomrule
        \\usepackage{multirow}        % \\multirow scenario labels
    """
    styles = _correct_and_majority_styles(
        pivot_df, correct_doses_dict, '#c8e6c9'  # solid light green (LaTeX-safe)
    )
    styler = (
        pivot_df.style
        .apply(lambda _: styles, axis=None)
        .format("{:.2f}")
    )
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    styler.to_latex(
        path,
        convert_css=True,   # translate background-color / font-weight to LaTeX
        hrules=True,        # booktabs \toprule, \midrule, \bottomrule
        caption=caption,
        label=label,
        position_float="centering",
        position="H"
    )
    print(f"Saved LaTeX table to {path}")
    return path


def _rec_mean_std(recs, n_levels, n_batches):
    """
    Recommendation %: each trial yields a single final dose, so the per-trial
    value is degenerate (0/100). Mean over all trials = the reported %. The std
    is the Monte-Carlo std of that proportion, estimated by splitting the trials
    into `n_batches` batches and taking the std of the batch percentages.
    """
    recs = np.asarray(recs)
    onehot = np.zeros((recs.size, n_levels))
    if recs.size:
        onehot[np.arange(recs.size), recs] = 100.0
    mean = onehot.mean(axis=0)
    nb = max(2, min(n_batches, recs.size))
    batch_means = np.array([b.mean(axis=0) for b in np.array_split(onehot, nb)])
    return mean, batch_means.std(axis=0, ddof=1)


def _alloc_mean_std(allocs_flat, n_cohorts, n_levels):
    """
    Allocation %: each trial yields a full allocation distribution, so the
    per-trial value (fraction of that trial's cohorts at each dose) is
    meaningful. Mean and std are taken across the per-trial percentages.
    """
    arr = np.asarray(allocs_flat).reshape(-1, n_cohorts)  # (n_trials, n_cohorts)
    frac = np.stack([(arr == k).mean(axis=1) for k in range(n_levels)], axis=1) * 100.0
    return frac.mean(axis=0), frac.std(axis=0, ddof=1)


def build_table2(rec_map, alloc_map, scenario_key, algos, n_cohorts, n_levels,
                 tox_probs, eff_probs, n_batches=5):
    """
    Reproduce Table 2 of the SEEDA paper: Recommended | Allocated side by side,
    one column per dose level, with toxicity / efficacy probability header rows
    and each cell showing "mean\\n(std)" over the trials.

    NOTE on std: the paper states "mean over 1000 repetitions, (standard
    deviation)" but does not define the std precisely, and its two halves have
    different magnitudes. We use the natural definition for each: a Monte-Carlo
    batch std for recommendation (a single trial gives only 0/100), and the
    across-trial std for allocation. Both reproduce the paper's magnitudes;
    `n_batches` is the one knob for the recommendation std.
    """
    doses = [f"Dose {k + 1}" for k in range(n_levels)]
    cols = pd.MultiIndex.from_product([["Recommended", "Allocated"], doses])
    index = ["Toxicity prob", "Efficacy prob"] + list(algos)
    table = pd.DataFrame(index=index, columns=cols, dtype=object)

    for half in ["Recommended", "Allocated"]:
        for k in range(n_levels):
            table.loc["Toxicity prob", (half, doses[k])] = f"{tox_probs[k]:g}"
            table.loc["Efficacy prob", (half, doses[k])] = f"{eff_probs[k]:g}"

    for algo in algos:
        r_mean, r_std = _rec_mean_std(rec_map[scenario_key][algo], n_levels, n_batches)
        a_mean, a_std = _alloc_mean_std(alloc_map[scenario_key][algo], n_cohorts, n_levels)
        for k in range(n_levels):
            table.loc[algo, ("Recommended", doses[k])] = f"{r_mean[k]:.2f}\n({r_std[k]:.2f})"
            table.loc[algo, ("Allocated", doses[k])] = f"{a_mean[k]:.2f}\n({a_std[k]:.2f})"
    return table


def _opt_col_body_style(opt_col):
    # Green + bold on the optimal-dose column body cells (used for on-screen display):
    def col_style(s):
        on = s.name[1] == opt_col
        return ['background-color: #c8e6c9; font-weight: bold' if on else '' for _ in s]
    return col_style


def _opt_col_header_style(opt_col):
    # Bold the optimal-dose column header (the paper bolds Dose 3):
    def header_style(vals):
        return ['font-weight: bold' if v == opt_col else '' for v in vals]
    return header_style


def style_table2(table, opt_dose):
    """On-screen styler: optimal-dose column highlighted green + bold (body and
    header), two-line mean/(std) cells, everything centred."""
    opt_col = f"Dose {opt_dose + 1}"
    return (
        table.style
        .apply(_opt_col_body_style(opt_col), axis=0)
        .apply_index(_opt_col_header_style(opt_col), axis="columns", level=1)
        .set_properties(**{"white-space": "pre-wrap", "text-align": "center"})
    )


def _parse_cell_value(cell):
    """Extract the numeric mean from a '\makecell{value \\ (std)}' string."""
    if isinstance(cell, str) and r"\makecell" in cell:
        try:
            inner = cell[len(r"\makecell{"):-1]          # "value \\ (std)"
            return float(inner.split(r" \\ ")[0].strip())
        except (ValueError, IndexError):
            return -np.inf
    try:
        return float(str(cell))
    except (ValueError, TypeError):
        return -np.inf


def export_table2_latex(table, opt_dose, path, caption=None, label=None):
    """
    Export the Table-2-format DataFrame to LaTeX with:
      - Green shading on the MTD (optimal-dose) column only.
      - Bold on the majority (most-recommended / most-allocated) cell per
        algorithm row, separately for the Recommended and Allocated halves.
      - Centered \\multicolumn headers with \\cmidrule underlines.
      - Vertical rule separating the two halves (l|cccccc|cccccc).
      - \\resizebox to keep the wide table within the text width.
      - Extra \\midrule after the Efficacy prob reference row.

    Requires in the document preamble:
        \\usepackage[table]{xcolor}   % \\cellcolor
        \\usepackage{booktabs}        % booktabs rules + \\cmidrule
        \\usepackage{makecell}        % two-line \\makecell cells
        \\usepackage{float}           % [H] placement
        \\usepackage{graphicx}        % \\resizebox
    """
    opt_col = f"Dose {opt_dose + 1}"
    prob_rows = {"Toxicity prob", "Efficacy prob"}

    # Turn "mean\n(std)" into \makecell{mean \\ (std)}:
    latex_tbl = table.map(
        lambda v: r"\makecell{" + v.replace("\n", r" \\ ") + "}"
        if isinstance(v, str) and "\n" in v else v
    )

    # Build per-cell style DataFrame:
    #   - green background  → MTD column (opt_col) for every row
    #   - bold              → argmax cell per algorithm row per half
    styles = pd.DataFrame('', index=latex_tbl.index, columns=latex_tbl.columns)

    for half in ["Recommended", "Allocated"]:
        half_cols = [c for c in latex_tbl.columns if c[0] == half]
        # Green on MTD column:
        mtd_key = (half, opt_col)
        if mtd_key in styles.columns:
            styles[mtd_key] += 'background-color: #c8e6c9; '
        # Bold on majority cell (algorithm rows only):
        for row in latex_tbl.index:
            if row in prob_rows:
                continue
            vals = {c: _parse_cell_value(latex_tbl.loc[row, c]) for c in half_cols}
            if vals:
                majority = max(vals, key=vals.get)
                styles.loc[row, majority] += 'font-weight: bold; '

    styler = (
        latex_tbl.style
        .apply(lambda _: styles, axis=None)
        .apply_index(_opt_col_header_style(opt_col), axis="columns", level=1)
    )

    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    # Generate raw LaTeX string:
    raw = styler.to_latex(
        convert_css=True,    # background-color -> \cellcolor, font-weight -> \bfseries
        hrules=True,         # booktabs rules
        caption=caption,
        label=label,
        position_float="centering",
        position="H",
    )

    # ── Post-process the generated LaTeX ──────────────────────────────────────

    # 1. Fix column spec: replace any all-l spec with l|cccccc|cccccc
    raw = re.sub(r'\\begin\{tabular\}\{[^}]+\}',
                 r'\\begin{tabular}{l|cccccc|cccccc}', raw)

    # 2. Center the section headers and add a vertical divider after Recommended:
    raw = raw.replace(r'\multicolumn{6}{r}{Recommended}',
                      r'\multicolumn{6}{c|}{Recommended}')
    raw = raw.replace(r'\multicolumn{6}{r}{Allocated}',
                      r'\multicolumn{6}{c}{Allocated}')

    # 3. Insert \cmidrule lines below the Recommended / Allocated header row:
    raw = raw.replace(
        r'\multicolumn{6}{c}{Allocated} \\',
        r'\multicolumn{6}{c}{Allocated} \\' + '\n'
        + r'\cmidrule(lr){2-7}\cmidrule(lr){8-13}'
    )

    # 4. Add \midrule after the Efficacy prob row to separate it from algorithms:
    lines = raw.split('\n')
    new_lines = []
    for line in lines:
        new_lines.append(line)
        if 'Efficacy prob' in line:
            new_lines.append(r'\midrule')
    raw = '\n'.join(new_lines)

    # 5. Wrap the tabular in \resizebox so the wide table fits the text width:
    raw = raw.replace(
        r'\begin{tabular}',
        r'\resizebox{\textwidth}{!}{%' + '\n' + r'\begin{tabular}'
    )
    raw = raw.replace(
        r'\end{tabular}',
        r'\end{tabular}' + '\n' + r'}%'
    )

    with open(path, 'w') as f:
        f.write(raw)
    print(f"Saved LaTeX table to {path}")
    return path

### Simulations

We run 2 main sets of simulations, differing in the number of cohorts used in their experiments. 

The more realistic setting uses around 10^1 cohorts while the setting for asymptotic behaviours uses more than 10^2.

#### Realistic

In [ ]:
N_REAL_COHORTS = 10

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [ ]:
a_algo_rec_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_rec_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_REAL_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
a_algo_alloc_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
a_algo_n_dle_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}

def add_simulations(dose_escalator, a, algo, n_efficate=0, n_cohorts=N_REAL_COHORTS):
    allocations, recommendations, n_dles = run_simulations(
        dose_escalator,
        dose_levels,
        lambda dose: dose_toxic_curve(dose, a),
        COHORT_SIZE,
        n_cohorts=n_cohorts,
        n_efficate=n_efficate,
        efficacy_env=EFFICACY_ENV
    )
    # Determine the final declared MTD for this trial:
    a_algo_rec_map[a_key(a)][algo].append(recommendations[-1])

    # Determine every cohort's allocated dose pooled across trials:
    a_algo_alloc_map[a_key(a)][algo].extend(allocations)

    # Determine the total toxicities during this trial:
    a_algo_n_dle_map[a_key(a)][algo].append(sum(n_dles))

    # Determine the recommendation at each cohort:
    for cohort, rec in enumerate(recommendations):
        cohort_algo_rec_map[a_key(a)][cohort_key(cohort)][algo].append(rec)

# Run the realistic simulation:
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0])
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1])
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=UCB_COEFF,
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2])
        
        # SEEDA (current version):
        seeda_dose_escalator = SEEDADoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_dose_escalator, a, ALGOS[3], N_EFFICATE)

        # SEEDA (UCB):
        seeda_original_dose_escalator = SEEDAOriginalDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_original_dose_escalator, a, ALGOS[4], N_EFFICATE)

        # SEEDA Plateau (UCB):
        seedapl_dose_escalator = SEEDAPlateauDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_dose_escalator, a, ALGOS[5], N_EFFICATE)

        # SEEDA Plateau (Modified L1):
        seedapl_fixed_dose_escalator = SEEDAPlateauFixedDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_fixed_dose_escalator, a, ALGOS[6], N_EFFICATE)

        # SEEDA Plateau (Paper):
        seedapl_naive_dose_escalator = SEEDAPlateauNaiveDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_naive_dose_escalator, a, ALGOS[7], N_EFFICATE)

/var/folders/z0/50y4smq97393y1f3vtzs_mcm0000gn/T/ipykernel_36603/3146517846.py:2: RuntimeWarning: overflow encountered in power
  return np.power((np.tanh(dose_levels) + 1) / 2, a_hat)


Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [ ]:
# Plot the dose recommendations:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_rec_map, 
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of MTD recommendations",
    img_path=f"plots/{timestamp}/Realistic/Recommendations/real_algo_dose_rec_proposals.png"
)

Plot the proposals made for each cohort, so that we can see the time evolution of our `DoseEscalator`'s proposals. This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [ ]:
for a in DOSE_LEVELS.keys():
    correct_mtds = {
        cohort_key(cohort): CORRECT_DOSES[a_key(a)]
        for cohort in range(N_REAL_COHORTS)
    }
    mtds = {
        cohort_key(cohort): CORRECT_MTDS[a_key(a)]
        for cohort in range(N_REAL_COHORTS)
    }
    plot_dose_proposals(
        N_LEVELS, 
        N_TRIALS, 
        cohort_algo_rec_map[a_key(a)], 
        correct_mtds, 
        mtds=mtds,
        unit_width=100,
        title_text="MTD recommendations at each cohort",
        show_fig=False,
        img_path=f"plots/{timestamp}/Realistic/Recommendations/real_{a_key(a)}_dose_rec_proposal_progression.png"
    )

Plot the distribution of dose limiting events across all the trial runs and cohorts.

In [ ]:
plot_n_dles(
    a_algo_n_dle_map, 
    img_path=f"plots/{timestamp}/Realistic/real_algo_n_dles.png"
)

Plot the distribution of dose allocations (the dose each cohort actually received), pooled across all cohorts and trial runs.

In [ ]:
plot_dose_proposals(
    N_LEVELS,
    N_TRIALS * N_REAL_COHORTS,
    a_algo_alloc_map,
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of dose allocations",
    img_path=f"plots/{timestamp}/Realistic/Allocations/real_algo_dose_allocations.png"
)

Make a table with the metrics.

In [ ]:
real_table = build_results_table(
    a_algo_rec_map, a_algo_alloc_map, OPTIMAL_DOSES, N_LEVELS, ALGOS
)

In [ ]:
# Paper-style wide layout (one column per dose level):
real_rec_pivot = real_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Rec (in %)", sort=False
)
real_alloc_pivot = real_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Alloc (in %)", sort=False
)

# Correct-dose summary: for each scenario/algorithm, the % recommending and
# allocating that algorithm's correct dose (efficacy-optimal for SEEDA/Plateau,
# toxicity MTD otherwise), plus whether that correct dose is the toxicity MTD.
correct_only = real_table[real_table["Is correct"]].copy()
correct_only["Is MTD"] = [
    dose == CORRECT_MTDS[scenario]
    for scenario, dose in zip(correct_only["Scenario"], correct_only["Dose"])
]
correct_summary = correct_only.set_index(["Scenario", "Algorithm"])[
    ["Dose", "Is MTD", "Rec (in %)", "Alloc (in %)"]
].rename(columns={
    "Dose": "Correct dose",
    "Rec (in %)": "Correct dose rec %",
    "Alloc (in %)": "Correct dose alloc %",
})

# Display doses as 1-6 (paper's numbering) rather than the internal 0-5 indices:
real_rec_pivot.columns = real_rec_pivot.columns + 1
real_alloc_pivot.columns = real_alloc_pivot.columns + 1
correct_summary["Correct dose"] = correct_summary["Correct dose"] + 1

print("Recommendation % by dose level")
display(highlight_correct_dose(real_rec_pivot, CORRECT_DOSES_DISPLAY))
print("\nAllocation % by dose level")
display(highlight_correct_dose(real_alloc_pivot, CORRECT_DOSES_DISPLAY))
print("\nCorrect-dose summary")
display(correct_summary)

In [ ]:
# Paper Table 2 layout (Recommended | Allocated side by side, mean over two lines
# with (std) beneath). Optimal biological dose (Dose 3) highlighted.
table2_real = build_table2(
    a_algo_rec_map, a_algo_alloc_map, a_key(A_STAR), ALGOS,
    N_REAL_COHORTS, N_LEVELS, TOXICITY_PROBS, EFFICACY_PROBS, n_batches=5,
)
print(f"Table 2 (realistic, n = {N_REAL_COHORTS} cohorts). "
      f"Optimal biological dose = Dose {OPTIMAL_DOSE + 1}.")
display(style_table2(table2_real, OPTIMAL_DOSE))

Table 2 (realistic, n = 10 cohorts). Optimal biological dose = Dose 3.


In [ ]:
# Export the realistic Table 2 to LaTeX:
export_table2_latex(
    table2_real, OPTIMAL_DOSE,
    f"plots/{timestamp}/Realistic/real_table2.tex",
    caption=f"Recommendation \\& allocation percentages (realistic, "
            f"n = {N_REAL_COHORTS} cohorts). "
            f"Green column: optimal biological dose (Dose {OPTIMAL_DOSE + 1}); "
            f"the toxicity MTD is Dose {TOX_MTD + 1}. "
            f"\\textbf{{Bold}}: majority recommended or allocated dose per algorithm. "
            f"Mean over {N_TRIALS} repetitions, (std).",
    label="tab:real_table2",
)

Export the realistic tables as LaTeX (recommendations and allocations folders).

In [ ]:
# Export the realistic tables to LaTeX, into the matching timestamp folders:
export_table_latex(
    real_rec_pivot, CORRECT_DOSES_DISPLAY,
    f"plots/{timestamp}/Realistic/Recommendations/real_rec_by_dose.tex",
    caption="Realistic: recommendation \\% by dose level "
            "(green = correct dose / MTD, bold = majority dose).",
    label="tab:real_rec_by_dose",
)
export_table_latex(
    real_alloc_pivot, CORRECT_DOSES_DISPLAY,
    f"plots/{timestamp}/Realistic/Allocations/real_alloc_by_dose.tex",
    caption="Realistic: allocation \\% by dose level "
            "(green = correct dose / MTD, bold = majority dose).",
    label="tab:real_alloc_by_dose",
)

#### Asymptotic

In [ ]:
N_ASYM_COHORTS = 300

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [ ]:
# Reinitialise the maps for the asymptotic setting:
a_algo_rec_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_rec_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_ASYM_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
a_algo_alloc_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
a_algo_n_dle_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}

# Run the asymptotic simulation:
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0], n_cohorts=N_ASYM_COHORTS)
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1], n_cohorts=N_ASYM_COHORTS)
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=UCB_COEFF,
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2], n_cohorts=N_ASYM_COHORTS)
        
        # SEEDA (current version):
        seeda_dose_escalator = SEEDADoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_dose_escalator, a, ALGOS[3], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        # SEEDA (UCB):
        seeda_original_dose_escalator = SEEDAOriginalDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_original_dose_escalator, a, ALGOS[4], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        # SEEDA Plateau (UCB):
        seedapl_dose_escalator = SEEDAPlateauDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_dose_escalator, a, ALGOS[5], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        # SEEDA Plateau (Modified L1):
        seedapl_fixed_dose_escalator = SEEDAPlateauFixedDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_fixed_dose_escalator, a, ALGOS[6], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        # SEEDA Plateau (Paper):
        seedapl_naive_dose_escalator = SEEDAPlateauNaiveDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_naive_dose_escalator, a, ALGOS[7], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

/var/folders/z0/50y4smq97393y1f3vtzs_mcm0000gn/T/ipykernel_36603/3146517846.py:2: RuntimeWarning:

overflow encountered in power



Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [ ]:
# Plot the dose recommendations:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_rec_map, 
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of MTD recommendations",
    img_path=f"plots/{timestamp}/Asymptotic/Recommendations/asym_algo_dose_rec_proposals.png",
)

Plot the distribution of dose allocations, pooled across all cohorts and trial runs.

In [ ]:
plot_dose_proposals(
    N_LEVELS,
    N_TRIALS * N_ASYM_COHORTS,
    a_algo_alloc_map,
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of dose allocations",
    img_path=f"plots/{timestamp}/Asymptotic/Allocations/asym_algo_dose_allocations.png",
)

Plot the proposal accuracy (whether it matches the MTD) for each cohort so that we can see the time evolution of our `DoseEscalator`'s proposals. 

This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [ ]:
plot_acc_progression(
    N_ASYM_COHORTS, 
    cohort_algo_rec_map,
    CORRECT_DOSES,
    show_fig=False, 
    img_path=f"plots/{timestamp}/Asymptotic/Recommendations/asym_dose_rec_proposal_acc_progression.png",
)

Make a table with the metrics.

In [ ]:
asym_table = build_results_table(
    a_algo_rec_map, a_algo_alloc_map, OPTIMAL_DOSES, N_LEVELS, ALGOS
)

In [ ]:
# Paper-style wide layout (one column per dose level):
asym_rec_pivot = asym_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Rec (in %)", sort=False
)
asym_alloc_pivot = asym_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Alloc (in %)", sort=False
)

# Correct-dose summary: for each scenario/algorithm, the % recommending and
# allocating that algorithm's correct dose (efficacy-optimal for SEEDA/Plateau,
# toxicity MTD otherwise), plus whether that correct dose is the toxicity MTD.
correct_only = asym_table[asym_table["Is correct"]].copy()
correct_only["Is MTD"] = [
    dose == CORRECT_MTDS[scenario]
    for scenario, dose in zip(correct_only["Scenario"], correct_only["Dose"])
]
correct_summary = correct_only.set_index(["Scenario", "Algorithm"])[
    ["Dose", "Is MTD", "Rec (in %)", "Alloc (in %)"]
].rename(columns={
    "Dose": "Correct dose",
    "Rec (in %)": "Correct dose rec %",
    "Alloc (in %)": "Correct dose alloc %",
})

# Display doses as 1-6 (paper's numbering) rather than the internal 0-5 indices:
asym_rec_pivot.columns = asym_rec_pivot.columns + 1
asym_alloc_pivot.columns = asym_alloc_pivot.columns + 1
correct_summary["Correct dose"] = correct_summary["Correct dose"] + 1

print("Recommendation % by dose level")
display(highlight_correct_dose(asym_rec_pivot, CORRECT_DOSES_DISPLAY))
print("\nAllocation % by dose level")
display(highlight_correct_dose(asym_alloc_pivot, CORRECT_DOSES_DISPLAY))
print("\nCorrect-dose summary")
display(correct_summary)

In [ ]:
# Paper Table 2 layout (Recommended | Allocated side by side, mean over two lines
# with (std) beneath). Optimal biological dose (Dose 3) highlighted.
table2_asym = build_table2(
    a_algo_rec_map, a_algo_alloc_map, a_key(A_STAR), ALGOS,
    N_ASYM_COHORTS, N_LEVELS, TOXICITY_PROBS, EFFICACY_PROBS, n_batches=5,
)
print(f"Table 2 (asymptotic, n = {N_ASYM_COHORTS} cohorts). "
      f"Optimal biological dose = Dose {OPTIMAL_DOSE + 1}.")
display(style_table2(table2_asym, OPTIMAL_DOSE))

Table 2 (asymptotic, n = 300 cohorts). Optimal biological dose = Dose 3.


In [ ]:
# Export the asymptotic Table 2 to LaTeX:
export_table2_latex(
    table2_asym, OPTIMAL_DOSE,
    f"plots/{timestamp}/Asymptotic/asym_table2.tex",
    caption=f"Recommendation \\& allocation percentages (asymptotic, "
            f"n = {N_ASYM_COHORTS} cohorts). "
            f"Green column: optimal biological dose (Dose {OPTIMAL_DOSE + 1}); "
            f"the toxicity MTD is Dose {TOX_MTD + 1}. "
            f"\\textbf{{Bold}}: majority recommended or allocated dose per algorithm. "
            f"Mean over {N_TRIALS} repetitions, (std).",
    label="tab:asym_table2",
)

Export the asymptotic tables as LaTeX (recommendations and allocations folders).

In [ ]:
# Export the asymptotic tables to LaTeX, into the matching timestamp folders:
export_table_latex(
    asym_rec_pivot, CORRECT_DOSES_DISPLAY,
    f"plots/{timestamp}/Asymptotic/Recommendations/asym_rec_by_dose.tex",
    caption="Asymptotic: recommendation \\% by dose level "
            "(green = correct dose / MTD, bold = majority dose).",
    label="tab:asym_rec_by_dose",
)
export_table_latex(
    asym_alloc_pivot, CORRECT_DOSES_DISPLAY,
    f"plots/{timestamp}/Asymptotic/Allocations/asym_alloc_by_dose.tex",
    caption="Asymptotic: allocation \\% by dose level "
            "(green = correct dose / MTD, bold = majority dose).",
    label="tab:asym_alloc_by_dose",
)